# 04 — Lead-Lag Analysis

**Project:** SERSA Product Demand Relationship Analysis  
**Author:** Cesar Enrique Banda Martinez  
**Date:** 2026  

---

## Context

Notebooks 02 and 03 identified which SKUs move together at the same point in time (contemporaneous correlation).  
This notebook asks a different and more valuable question:

> **Does the demand of one SKU anticipate the demand of another by one or more months?**

This is known as **lead-lag analysis**. If SKU A at month `t` correlates with SKU B at month `t+k`, then A *leads* B by `k` months — meaning A's behavior today predicts B's behavior `k` months from now.

### Why this matters operationally
A confirmed lead-lag relationship would allow SERSA to use early movement in one product as a forward signal for replenishment planning of another — even before demand materializes.

### Method: cross-correlation at lags 1–6
We compute `corr(A_t, B_{t+k})` for lags k = 1, 2, 3, 4, 5, 6 months.  
We work on **growth rates** (from notebook 03) rather than raw counts, for the same detrending reason established previously.

We focus on the **104 pairs with growth-rate r ≥ 0.75** identified in notebook 03 as our candidate set — pairs that already show genuine contemporaneous co-movement are the most likely to also exhibit lead-lag structure.

### Interpretation
- **Lag 0**: contemporaneous (already measured in notebook 03).
- **Lag +k**: SKU_A leads SKU_B by k months.
- **Lag −k**: SKU_B leads SKU_A by k months (equivalent to SKU_A lagging SKU_B).

The lag with the **highest absolute cross-correlation** for a given pair identifies the dominant temporal relationship.

---

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

---
## 2. Configuration

In [ ]:
DATA_DIR       = os.path.join(os.getcwd(), "..", "outputs", "tables")
OUTPUT_TABLES  = os.path.join(os.getcwd(), "..", "outputs", "tables")
OUTPUT_FIGURES = os.path.join(os.getcwd(), "..", "outputs", "figures")

MAX_LAG        = 6     # months
CORR_THRESHOLD = 0.75  # same threshold used in notebooks 02 and 03

print(f"Data dir:       {os.path.normpath(DATA_DIR)}")
print(f"Output tables:  {os.path.normpath(OUTPUT_TABLES)}")
print(f"Output figures: {os.path.normpath(OUTPUT_FIGURES)}")
print(f"Max lag:        {MAX_LAG} months")

---
## 3. Load Data

In [ ]:
# Filtered pivot (86 SKUs)
pivot = pd.read_csv(
    os.path.join(DATA_DIR, "02_pivot_filtered.csv"),
    index_col="Month",
    parse_dates=True,
    decimal=","
)

# Growth rates
growth = pivot.pct_change()
growth.replace([np.inf, -np.inf], np.nan, inplace=True)

# Top pairs from notebook 03 (genuine co-movement candidates)
top_pairs = pd.read_csv(
    os.path.join(DATA_DIR, "03_top_positive_pairs_growth.csv"),
    decimal=","
)

print(f"Pivot:      {pivot.shape[0]} months x {pivot.shape[1]} SKUs")
print(f"Top pairs:  {len(top_pairs)} pairs with growth-rate r >= {CORR_THRESHOLD}")
print()
print(top_pairs.head(5).to_string(index=False))

---
## 4. Cross-Correlation Function

For each candidate pair, we compute Pearson correlation at lags −6 to +6.  
A positive lag `k` means SKU_A leads SKU_B: the value of A at time `t` is correlated with B at time `t+k`.

In [ ]:
def cross_corr_pair(series_a, series_b, max_lag):
    """
    Compute Pearson cross-correlation between series_a and series_b
    at integer lags from -max_lag to +max_lag.

    Positive lag k: series_a leads series_b by k periods.
    Negative lag k: series_b leads series_a by |k| periods.

    Parameters
    ----------
    series_a, series_b : pd.Series
        Time series of equal length (NaN values handled via pairwise dropna).
    max_lag : int
        Maximum lag to compute in both directions.

    Returns
    -------
    dict : {lag: pearson_r}
    """
    results = {}
    for lag in range(-max_lag, max_lag + 1):
        if lag == 0:
            combined = pd.concat([series_a, series_b], axis=1).dropna()
        elif lag > 0:
            # a leads b: align a[t] with b[t+lag]
            combined = pd.concat([series_a.iloc[:-lag], series_b.iloc[lag:].values], axis=1).dropna()
        else:
            # b leads a: align b[t] with a[t+|lag|]
            combined = pd.concat([series_a.iloc[-lag:].values, series_b.iloc[:lag]], axis=1).dropna()

        if len(combined) < 5:
            results[lag] = np.nan
        else:
            results[lag] = combined.iloc[:, 0].corr(combined.iloc[:, 1])
    return results

print("cross_corr_pair() defined.")

---
## 5. Compute Cross-Correlations for All Candidate Pairs

In [ ]:
records = []

for _, row in top_pairs.iterrows():
    sku_a = row["SKU_A"]
    sku_b = row["SKU_B"]

    if sku_a not in growth.columns or sku_b not in growth.columns:
        continue

    cc = cross_corr_pair(growth[sku_a], growth[sku_b], MAX_LAG)

    for lag, r in cc.items():
        records.append({
            "SKU_A"    : sku_a,
            "SKU_B"    : sku_b,
            "lag"      : lag,
            "Pearson_r": round(r, 4) if not np.isnan(r) else np.nan
        })

cc_df = pd.DataFrame(records)

print(f"Cross-correlation records: {len(cc_df):,}")
print(f"Pairs computed:            {cc_df.groupby(['SKU_A','SKU_B']).ngroups}")
print()
print(cc_df.head(13).to_string(index=False))

---
## 6. Dominant Lag per Pair

For each pair, identify the lag at which the absolute cross-correlation is highest.  
Pairs where the dominant lag is 0 are purely contemporaneous.  
Pairs where the dominant lag is ≠ 0 have a potential predictive structure.

In [ ]:
dominant = (
    cc_df.dropna(subset=["Pearson_r"])
    .loc[cc_df.groupby(["SKU_A", "SKU_B"])["Pearson_r"].apply(lambda x: x.abs().idxmax())]
    .rename(columns={"Pearson_r": "max_abs_r", "lag": "dominant_lag"})
    [["SKU_A", "SKU_B", "dominant_lag", "max_abs_r"]]
    .sort_values("max_abs_r", ascending=False)
    .reset_index(drop=True)
)

print("Dominant lag distribution:")
print(dominant["dominant_lag"].value_counts().sort_index().to_string())
print()
print("Top 20 pairs by max absolute cross-correlation:")
print(dominant.head(20).to_string(index=False))

---
## 7. Lead-Lag Candidates

Pairs where the dominant lag is **not zero** are the most analytically interesting — they suggest one SKU anticipates the other.

In [ ]:
lead_lag_candidates = dominant[dominant["dominant_lag"] != 0].copy()
contemporaneous     = dominant[dominant["dominant_lag"] == 0].copy()

print(f"Contemporaneous pairs (dominant lag = 0) : {len(contemporaneous)}")
print(f"Lead-lag candidates   (dominant lag != 0): {len(lead_lag_candidates)}")
print()
if len(lead_lag_candidates) > 0:
    print(lead_lag_candidates.to_string(index=False))
else:
    print("No lead-lag structure found above threshold.")

---
## 8. Cross-Correlation Profile — Top Pairs

Visualizing the full cross-correlation profile (lag −6 to +6) for the top pairs shows the shape of the temporal relationship.

In [ ]:
# Select top N pairs to visualize (by contemporaneous r from notebook 03)
N_PAIRS = min(6, len(top_pairs))
pairs_to_plot = top_pairs.head(N_PAIRS)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, (_, row) in enumerate(pairs_to_plot.iterrows()):
    sku_a = row["SKU_A"]
    sku_b = row["SKU_B"]

    pair_cc = cc_df[(cc_df["SKU_A"] == sku_a) & (cc_df["SKU_B"] == sku_b)].sort_values("lag")

    ax = axes[i]
    colors = ["#D7191C" if r < 0 else "#2C7BB6" for r in pair_cc["Pearson_r"].fillna(0)]
    ax.bar(pair_cc["lag"], pair_cc["Pearson_r"].fillna(0), color=colors, alpha=0.85)
    ax.axhline(y=0, color="black", linewidth=0.8)
    ax.axvline(x=0, color="gray",  linewidth=0.8, linestyle="--", alpha=0.5)
    ax.set_ylim(-1, 1)
    ax.set_xticks(range(-MAX_LAG, MAX_LAG + 1))
    ax.set_xlabel("Lag (months)", fontsize=9)
    ax.set_ylabel("Pearson r", fontsize=9)
    ax.set_title(f"{sku_a}\n\u2194 {sku_b}", fontsize=9, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    sns.despine(ax=ax)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Cross-Correlation Profiles — Top Pairs (Growth Rates)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "04_crosscorr_profiles_top_pairs.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 04_crosscorr_profiles_top_pairs.png")

---
## 9. Dominant Lag Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

lag_counts = dominant["dominant_lag"].value_counts().sort_index()
colors = ["#D7191C" if lag != 0 else "#2C7BB6" for lag in lag_counts.index]
ax.bar(lag_counts.index, lag_counts.values, color=colors, alpha=0.85)

ax.set_xlabel("Dominant Lag (months)", fontsize=11)
ax.set_ylabel("Number of Pairs", fontsize=11)
ax.set_title("Distribution of Dominant Lag Across All Candidate Pairs", fontsize=13, fontweight="bold")
ax.set_xticks(range(-MAX_LAG, MAX_LAG + 1))
ax.grid(axis="y", alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2C7BB6", alpha=0.85, label="Contemporaneous (lag = 0)"),
    Patch(facecolor="#D7191C", alpha=0.85, label="Lead-lag structure (lag \u2260 0)"),
]
ax.legend(handles=legend_elements, fontsize=9)
sns.despine()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "04_dominant_lag_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 04_dominant_lag_distribution.png")

---
## 10. Export

In [ ]:
cc_df.to_csv(
    os.path.join(OUTPUT_TABLES, "04_crosscorr_all_pairs.csv"),
    index=False,
    decimal=","
)

dominant.to_csv(
    os.path.join(OUTPUT_TABLES, "04_dominant_lag_per_pair.csv"),
    index=False,
    decimal=","
)

lead_lag_candidates.to_csv(
    os.path.join(OUTPUT_TABLES, "04_lead_lag_candidates.csv"),
    index=False,
    decimal=","
)

print("Export complete.")
print(f"  04_crosscorr_all_pairs.csv      -> {len(cc_df):,} records")
print(f"  04_dominant_lag_per_pair.csv    -> {len(dominant)} pairs")
print(f"  04_lead_lag_candidates.csv      -> {len(lead_lag_candidates)} pairs with lag != 0")

---
## 11. Summary

| Output | Description |
|--------|-------------|
| `04_crosscorr_all_pairs.csv` | Full cross-correlation at each lag for all candidate pairs |
| `04_dominant_lag_per_pair.csv` | Dominant lag and max absolute r per pair |
| `04_lead_lag_candidates.csv` | Pairs where dominant lag ≠ 0 |
| `04_crosscorr_profiles_top_pairs.png` | Cross-correlation bar charts for top 6 pairs |
| `04_dominant_lag_distribution.png` | Histogram of dominant lags across all pairs |

### Key questions this notebook answers
- Are any demand relationships truly predictive (lead-lag), or is all co-movement contemporaneous?
- Which SKU pairs show the clearest lead-lag structure?
- What is the typical lag length when lead-lag exists?

**Next:** `05_product_network.ipynb` — build a product demand network using the correlation matrices from notebooks 02 and 03, with nodes as SKUs and edges weighted by correlation strength.